In [3]:
from one.api import ONE
from brainbox.io.one import SessionLoader
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from collections import defaultdict
import pandas as pd
from ibl_info.utils import check_config
import numpy as np
from matplotlib import pyplot as plt
from glob import glob
import pickle as pkl
from scipy import stats
from statsmodels.stats.multitest import multipletests
import seaborn as sns
from manifold.utils import *
from glob import glob

In [4]:
%load_ext autoreload 
%autoreload 2

In [5]:
# load widefield
# check which regions carry significant information about stimulus and choice

In [45]:
# for each animal
# we load the three tables
# look at differences in accuracy

In [46]:
one = ONE(mode="local")
sessions_all = one.search(datasets="widefieldU.images.npy")
sessions_all = np.asarray([str(s) for s in sessions_all])  # type: ignore

In [ ]:
# copy over
all_regions = regions

In [ ]:
global_accuracies = {}
for rx in all_regions:
    accuracies = []
    for idx, eid in enumerate(sessions_all):  # type: ignore
        try:
            data_delta = pkl.load(open(f"../data/generated/wifi/diff_stim/{eid}_stim.pkl", "rb"))
            data_stim_zero = pkl.load(
                open(f"../data/generated/wifi/stim_intervals/stim1/{eid}_stim-1.pkl", "rb")
            )
            data_stim_one = pkl.load(
                open(f"../data/generated/wifi/stim_intervals/stim2/{eid}_stim-2.pkl", "rb")
            )
            accuracies.append(
                [
                    data_delta[rx]["mean_score"],
                    data_stim_zero[rx]["mean_score"],
                    data_stim_one[rx]["mean_score"],
                ]
            )
        except Exception as e:
            print(e)
            print(eid, idx)
            accuracies.append([0, 0, 0])
    global_accuracies[rx] = accuracies

In [95]:
def get_star_rating(p):
    if p >= 0.05:
        return "ns"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    else:
        return "*"


def check_significance(df):
    columns = df.columns.tolist()
    pairs = [
        (columns[0], columns[1]),
        (columns[1], columns[2]),
        (columns[0], columns[2]),
    ]
    sig_flag_all = []
    for col1, col2 in pairs:
        # Wilcoxon signed-rank test since data is paired across epochs per animal
        _, p_raw = stats.wilcoxon(df[col1], df[col2], alternative="two-sided")
        test_name = "Wilcoxon"

        p_corrected = min(1.0, p_raw * 3)  # type: ignore
        # status = "SIGNIFICANT" if p_corrected < 0.05 else "NOT SIGNIFICANT"
        sig_flag = get_star_rating(p_corrected)
        sig_flag_all.append(sig_flag)

    return sig_flag_all

In [ ]:
for region_name in global_accuracies.keys():
    w = np.asarray(global_accuracies[region_name])
    df = pd.DataFrame(w, columns=["delta", "0", "1"])
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(df, ax=ax)
    sns.stripplot(df, ax=ax)
    sns.despine()
    ax.set_ylim(0, 0.8)

    result = check_significance(df)
    ax.set_title(f"{region_name}, {result}")

In [100]:
# let's use the activity at frame 1

In [104]:
dfs.to_parquet("../data/generated/wifi/stimulus_accuracy_aggregates.pqt")

In [102]:
dfs = pd.DataFrame(global_accuracies)

In [103]:
dfs

,MOB,MOp,MOs,SSp-n,SSp-m,SSp-un,PL,RSPv,RSPd,RSPagl,...,AUDd,AUDp,VISpor,VISpm,VISl,VISal,VISrl,VISa,VISam,VISp
0,"[0.5935023041474654, 0.5501740911418331, 0.590...","[0.5592985151049668, 0.582816180235535, 0.5942...","[0.5741628264208909, 0.6040860215053765, 0.556...","[0.49661546338965684, 0.5349411162314388, 0.54...","[0.5385970302099334, 0.5057962109575012, 0.521...","[0.5222478238607271, 0.556236559139785, 0.5309...","[0.5254992319508448, 0.5206810035842293, 0.507...","[0.5258883768561188, 0.5313671274961597, 0.550...","[0.518294930875576, 0.5436917562724014, 0.5630...","[0.5110547875064004, 0.5189400921658985, 0.519...",...,"[0.5191244239631336, 0.5036763952892984, 0.511...","[0.4956118791602663, 0.5086994367639528, 0.510...","[0.4964260112647209, 0.5474244751664107, 0.499...","[0.4788120839733743, 0.5176907322068612, 0.500...","[0.5091449052739375, 0.5165284178187404, 0.512...","[0.5364004096262162, 0.5016180235535075, 0.502...","[0.4519662058371735, 0.5100716845878137, 0.510...","[0.5487711213517665, 0.5344034818228367, 0.566...","[0.45909882232462873, 0.5301484895033282, 0.53...","[0.5891909882232462, 0.604347158218126, 0.5630..."
1,"[0.5987753153424794, 0.553166490778431, 0.5241...","[0.5296572692095081, 0.5259173827830544, 0.537...","[0.5363219257249108, 0.5675305291723202, 0.522...","[0.513717774762551, 0.4995457058143626, 0.5246...","[0.5263897683300669, 0.501486506859641, 0.4990...","[0.5531112116186743, 0.5249982411176441, 0.524...","[0.533268505955073, 0.5399703502688578, 0.5056...","[0.4777285290718126, 0.47914417809940196, 0.47...","[0.5075576662143827, 0.5396783757977788, 0.521...","[0.5089275843007186, 0.5477802904668576, 0.507...",...,"[0.556814915322378, 0.5631443791145283, 0.5122...","[0.575601286496809, 0.5101150811598572, 0.4954...","[0.5317855168601437, 0.49759183878586855, 0.49...","[0.5648831599577868, 0.44876124428363234, 0.49...","[0.4946957133524298, 0.48203678576812903, 0.50...","[0.5999552741343785, 0.5213282074476104, 0.466...","[0.5668480828182322, 0.5361430222624253, 0.521...","[0.5625951052816724, 0.5202070455801799, 0.491...","[0.5532805668626564, 0.4955836976732499, 0.495...","[0.5514287150108046, 0.5088687873762501, 0.539..."
2,"[0.5764968317599897, 0.5496831759989654, 0.586...","[0.6373787663261348, 0.6191258243889822, 0.629...","[0.5950892279839648, 0.5721970774602354, 0.584...","[0.6149198241303504, 0.5091394025604552, 0.547...","[0.616927453769559, 0.5615511444458813, 0.5953...","[0.557238458554248, 0.5781747058062848, 0.5464...","[0.5125824388982283, 0.5501648777964567, 0.488...","[0.5273664813138497, 0.5610694426483901, 0.555...","[0.6000614250614251, 0.5431527221000906, 0.609...","[0.6126664942454416, 0.5744180783654468, 0.595...",...,"[0.5345435148066727, 0.5638012414328204, 0.605...","[0.5361890598732704, 0.5427033492822967, 0.568...","[0.5349734902366482, 0.5059776283460493, 0.518...","[0.6253911806543385, 0.555447433079012, 0.5763...","[0.6260894866158024, 0.5184663132031553, 0.534...","[0.5864476917108495, 0.5355230828915041, 0.565...","[0.6134520884520883, 0.572051597051597, 0.5958...","[0.5734870037501617, 0.538035044613992, 0.5766...","[0.6335510151299625, 0.5822610888400362, 0.617...","[0.6117127893443683, 0.48902431139273245, 0.51..."
3,"[0.6146685089499465, 0.5922652887922969, 0.637...","[0.6110786232770573, 0.6119926644501696, 0.629...","[0.6297943263432116, 0.6147328929124869, 0.621...","[0.5783715463185518, 0.5411259277383575, 0.577...","[0.6154474013730769, 0.5846124488454544, 0.630...","[0.5528360367880942, 0.5880715515959258, 0.518...","[0.5635998167311945, 0.5385827852060815, 0.582...","[0.5820351473106982, 0.5519059236124104, 0.601...","[0.6144762925968038, 0.5726116284536815, 0.611...","[0.574491956802295, 0.5943934138372746, 0.6179...",...,"[0.5602719047002211, 0.5695246045567726, 0.557...","[0.5729746158311624, 0.5613664080753417, 0.554...","[0.5892598243114227, 0.5625472444911411, 0.588...","[0.5820581038875056, 0.5372843593028109, 0.593...","[0.572025408156